# LLM-разметка с ablation по графовым признакам

Эксперимент: подаём в промпт LLM структурные сигналы из графа и сравниваем 5 конфигураций (от `no_graph` до полного набора фич).

**Итог:** графовые признаки не улучшили F1 относительно базовой разметки без графа. Подробности — в [`../README.md`](../README.md).

Требуется `OLLAMA_API_KEY` и артефакты из `04_graph_build_and_features.ipynb`.

In [1]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
ART_DIR = ROOT / "artifacts"
DATA_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)
%pip install pandas
%pip install openpyxl
%pip install ollama
%pip install transformers
%pip install torch
%pip install accelerate


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.12.0-py3-none-any.whl (380 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import importlib

mods = ['ollama', 'transformers']
for m in mods:
    try:
        importlib.import_module(m)
        print(m, 'OK')
    except Exception as e:
        print(m, 'ERR', type(e).__name__ + ':', e)

ollama OK


/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers OK


In [2]:
import os
import json
import re
import time
from dataclasses import dataclass
from datetime import datetime
from typing import Any, Dict, Optional, Tuple

import pandas as pd

INPUT_XLSX = 'markup_dataset_filled_20260220.xlsx'
SHEET = 0
START_ROW = 0
MAX_ROWS = 0  # 0 = all

# Cloud API mode:
OLLAMA_CLOUD_HOST = 'https://ollama.com'
OLLAMA_API_KEY = os.getenv('OLLAMA_API_KEY')

# Safety limits for calls
CLOUD_TIMEOUT_S = 60
NUM_PREDICT = 1024

# Retry policy (helps with occasional empty responses / rate limits)
RETRY_ATTEMPTS = 3
RETRY_BACKOFF_S = 1.0

# Throttle between cloud requests (avoid rate limits)
CLOUD_THROTTLE_S = 0.25

# paths resolved from narratives_graph root (see first cell)
input_path = str(DATA_DIR / 'markup_dataset_filled_20260220.xlsx')
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
name, ext = os.path.splitext(os.path.basename(input_path))
out_path = str(DATA_DIR / f'{name}_filled_{stamp}{ext}')


In [ ]:
def build_prompt(message: str, topic: str, graph_signals) -> str:
    graph_block = format_graph_block(graph_signals or {})
    prompt = ('Ты — эксперт по экономическим новостям и общественному восприятию в России. '
        'Твоя задача — определить, является ли короткая новость экономическим нарративом для широкой российской аудитории.\n\n'
        'Критерии нарратива в вашем понимании:\n\n'
        '1. Релевантность для России: Новость должна иметь прямой или косвенный экономический эффект для жителей России. '
        'Мировые новости без последствий для России — не нарратив.\n\n'
        '2. Широкий общественный резонанс: Фокус на обычных гражданах, а не на узких группах. '
        'Нарратив — это яркая новость, которая может вызвать сильный отклик в массах, желание делиться ею '
        'и влиять на импульсивные решения (например, срочные покупки, вывод средств).\n\n'
        '3. Яркость и сила события: В основе нарратива лежит сильное событие (резкий рост цен, важное политическое заявление, '
        'масштабные санкции), а не рутинная информация.\n\n'
        'Проанализируй новость по следующему плану (рассуждения держи в уме, в ответ не выноси):\n\n'
        '[1] Триггер и релевантность:\n'
        'В чём суть новости? Есть ли чёткое триггерное событие (заявление, решение, кризис)?\n'
        'Имеет ли это событие прямые последствия для экономического положения или настроений широких слоёв населения России?\n\n'
        '[2] Эмоциональный заряд и упрощение:\n'
        'Какие эмоции может вызвать текст у обычного человека? (тревога, страх, гнев, оптимизм).\n'
        'Сводится ли основная мысль к простым, обобщающим формулировкам? (Например: «Цены на всё вырастут», «Рубль обвалится», «Наступит дефицит»).\n\n'
        '[3] Логика воздействия и аудитория:\n'
        'Пытается ли новость объяснить, как именно это событие повлияет на жизнь людей, а не просто констатирует факт?\n'
        'Направлена ли новость на массовую аудиторию, а не на профессионалов?\n\n'
        '[4] Источники и резонансный потенциал:\n'
        'Кто является источником или героем новости? (Правительство, ЦБ, известный политик, эксперты в СМИ). '
        'Усиливает ли это авторитетность и потенциальное распространение?\n'
        'Может ли эта новость стать «вирусной» историей для обсуждения в соцсетях и бытовых разговорах?\n\n'
        'Пояснения к полям:\n\n'
        'economic_narrative (Да/Нет): Итоговое решение. Да — если есть триггерное событие, релевантное для широкой российской аудитории, '
        'и новость обладает потенциалом вызвать эмоциональный отклик и массовое обсуждение.\n\n'
        'narrative_strength (1-3): Сила нарративных свойств. Оценивай, насколько текст эмоционален, упрощён и побуждает к действию. '
        '1 — констатация факта; 3 — прямое предупреждение о катастрофических последствиях для всех. '
        'Не нарративные новости в большинстве своем должны получать оценку 1, при этом нарративы тоже могут изредка получать такую оценку.\n\n'
        'economic_effect (-2..2): Влияние на экономическое положение обычного человека в России. Примеры: крах банков: -2, рост в отдельном регионе: -1, отмена санкций: 2.\n\n'
        'topic_agreement (1-3): Твоя оценка правильности выбранной темы (topic). Считай, что тема выбрана из фиксированного списка.\n\n'
        'information_resonance (1-3): Потенциал широкого и эмоционального восприятия в обществе. Высокий резонанс — темы, затрагивающие каждого '
        '(цены на еду, бензин, рубль, важные политические решения для всего населения).\n\n'
        'ВАЖНО: Ответь строго одним JSON-объектом БЕЗ пояснений, БЕЗ markdown и БЕЗ текста вокруг.\n'
        'Схема JSON (ключи строго такие):\n'
        '{\n'
        '  "economic_effect": -2,\n'
        '  "information_resonance": 1,\n'
        '  "topic_agreement": 1,\n'
        '  "economic_narrative": "Да",\n'
        '  "narrative_strength": 1,\n'
        '  "comment": "..."\n'
        '}\n\n'
        f'Тема (topic): {topic}\n'
        f'Новость (message): {message}\n'
        f'{graph_block}'
        )
        
    return prompt

def _to_int(v: Any) -> int:
    if isinstance(v, bool):
        raise ValueError('Boolean is not allowed')
    if isinstance(v, (int, float)):
        return int(v)
    s = _strip(v)
    if not s:
        raise ValueError('Empty number')
    return int(float(s))


def _validate_payload(obj: Dict[str, Any]) -> Dict[str, Any]:
    required = {
        'economic_effect',
        'information_resonance',
        'topic_agreement',
        'economic_narrative',
        'narrative_strength',
        'comment',
    }
    missing = sorted(required - set(obj.keys()))
    if missing:
        raise ValueError('Missing keys: ' + ', '.join(missing))

    payload = {
        'economic_effect': _to_int(obj.get('economic_effect')),
        'information_resonance': _to_int(obj.get('information_resonance')),
        'topic_agreement': _to_int(obj.get('topic_agreement')),
        'narrative_strength': _to_int(obj.get('narrative_strength')),
        'economic_narrative': _strip(obj.get('economic_narrative')),
        'comment': _strip(obj.get('comment')),
    }

    if payload['economic_narrative'] not in {'Да', 'Нет'}:
        raise ValueError('economic_narrative must be "Да" or "Нет"')

    if payload['economic_effect'] < -2 or payload['economic_effect'] > 2:
        raise ValueError('economic_effect out of range (-2..2)')

    for k in ['information_resonance', 'topic_agreement', 'narrative_strength']:
        if payload[k] < 1 or payload[k] > 3:
            raise ValueError(f'{k} out of range (1..3)')

    if not payload['comment']:
        payload['comment'] = ''

    return payload

def _strip(x: Any) -> str:
    if x is None:
        return ''
    return str(x).strip()


def _normalize_ollama_model_name(model: str, host: Optional[str]) -> str:
    if host and model.endswith('-cloud'):
        return model[:-6]
    return model


def _ollama_message_content(resp: Any) -> str:
    if resp is None:
        return ''

    if isinstance(resp, dict):
        msg = resp.get('message') or {}
        if isinstance(msg, dict):
            return _strip(msg.get('content'))

    if hasattr(resp, 'message') and hasattr(resp.message, 'content'):
        return _strip(resp.message.content)

    try:
        msg = resp['message']
        return _strip(msg['content'])
    except Exception:
        return _strip(str(resp))


@dataclass(frozen=True)
class ModelSpec:
    name: str
    provider: str


def _ollama_client(host: str):
    from ollama import Client

    api_key = os.getenv('OLLAMA_API_KEY')
    if not api_key:
        raise ValueError('OLLAMA_API_KEY is not set')

    headers = {'Authorization': f'Bearer {api_key}'}
    timeout = CLOUD_TIMEOUT_S

    return Client(host=host, headers=headers, timeout=timeout)


def _should_retry_exc(e: Exception) -> bool:
    status = getattr(e, 'status_code', None)
    if isinstance(status, int) and status in {408, 409, 425, 429, 500, 502, 503, 504}:
        return True

    msg = str(e).lower()
    if 'empty model response' in msg:
        return True
    if 'timed out' in msg or 'timeout' in msg:
        return True
    if 'connection' in msg or 'network' in msg:
        return True

    return False


def infer_ollama(model: str, prompt: str, host: Optional[str]) -> str:
    last_err: Optional[Exception] = None

    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            if host:
                client = _ollama_client(host)
                resp = client.chat(
                    model=_normalize_ollama_model_name(model, host),
                    messages=[{'role': 'user', 'content': prompt}],
                    options={'temperature': 0.2, 'num_predict': NUM_PREDICT},
                )
            else:
                from ollama import chat

                resp = chat(
                    model=_normalize_ollama_model_name(model, host=None),
                    messages=[{'role': 'user', 'content': prompt}],
                    options={'temperature': 0.2, 'num_predict': NUM_PREDICT},
                )

            text = _ollama_message_content(resp)
            if not text:
                raise ValueError('Empty model response')
            return text
        except Exception as e:
            last_err = e
            if attempt < RETRY_ATTEMPTS and _should_retry_exc(e):
                time.sleep(RETRY_BACKOFF_S * attempt)
                continue
            break

    raise last_err or ValueError('Empty model response')


_WEDLM_LLM = None
_WEDLM_TOKENIZER = None

def _json_from_text(text: str) -> Dict[str, Any]:
    text = _strip(text)
    if not text:
        raise ValueError('Empty text')

    try:
        obj = json.loads(text)
        if not isinstance(obj, dict):
            raise ValueError('JSON root is not an object')
        return obj
    except Exception:
        pass

    m = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if not m:
        raise ValueError('No JSON object found in model output')

    obj = json.loads(m.group(0))
    if not isinstance(obj, dict):
        raise ValueError('JSON root is not an object')
    return obj


def _extract_text(obj: Any) -> str:
    if obj is None:
        return ''
    if isinstance(obj, str):
        return obj
    if isinstance(obj, dict):
        return _strip(obj.get('text') or obj.get('content') or '')
    if hasattr(obj, 'text'):
        return _strip(getattr(obj, 'text'))
    return _strip(str(obj))


def infer_wedlm(prompt: str) -> str:
    _init_wedlm()

    from wedlm import SamplingParams

    messages = [{'role': 'user', 'content': prompt}]

    text = _WEDLM_TOKENIZER.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    outs = _WEDLM_LLM.generate([text], SamplingParams(temperature=0.2, max_tokens=512))
    if not outs:
        raise ValueError('Empty model response')

    first = outs[0]
    extracted = _extract_text(first)
    if extracted:
        return extracted

    if isinstance(first, dict):
        inner = first.get('outputs')
        if isinstance(inner, list) and inner:
            extracted = _extract_text(inner[0])
            if extracted:
                return extracted

    raise ValueError('Empty model response')


def run_inference(
    model_spec: ModelSpec, prompt: str, cloud_host: Optional[str]
) -> Tuple[Optional[Dict[str, Any]], Optional[str], Optional[float]]:
    try:
        t0 = None
        if model_spec.provider in {'local_ollama', 'wedlm'}:
            t0 = time.perf_counter()

        if model_spec.provider == 'cloud_ollama':
            text = infer_ollama(model_spec.name, prompt, host=cloud_host)
        elif model_spec.provider == 'local_ollama':
            text = infer_ollama(model_spec.name, prompt, host=None)
        elif model_spec.provider == 'wedlm':
            text = infer_wedlm(prompt)
        else:
            raise ValueError(f'Unknown provider: {model_spec.provider}')

        elapsed_s = (time.perf_counter() - t0) if t0 is not None else None

        last_err: Optional[Exception] = None
        for attempt in range(1, RETRY_ATTEMPTS + 1):
            try:
                payload = _validate_payload(_json_from_text(text))
                return payload, None, elapsed_s
            except Exception as e:
                last_err = e
                if attempt < RETRY_ATTEMPTS:
                    time.sleep(RETRY_BACKOFF_S * attempt)

        raise last_err or ValueError('Invalid JSON')

    except Exception as e:
        elapsed_s = None
        try:
            if 't0' in locals() and t0 is not None:
                elapsed_s = time.perf_counter() - t0
        except Exception:
            elapsed_s = None
        return None, f'{type(e).__name__}: {e}', elapsed_s


def col(name: str, idx: int) -> str:
    return f'{name} ({idx})'


def ensure_columns(df: pd.DataFrame, n: int) -> pd.DataFrame:
    for c in ['message', 'topic']:
        if c not in df.columns:
            raise ValueError(f'Missing required column: {c}')

    metric_cols = [
        'LLm',
        'Экономический эффект',
        'Информационный резонанс',
        'Правильность определения темы',
        'Экономический нарратив',
        'Сила нарратива',
        'Комментарий',
    ]

    for i in range(1, n + 1):
        for m in metric_cols:
            cn = col(m, i)
            if cn not in df.columns:
                df[cn] = ''

    for i in range(1, n + 1):
        for m in metric_cols:
            cn = col(m, i)
            df[cn] = df[cn].astype('object').where(~pd.isna(df[cn]), '')

    return df


def run_slot(
    df: pd.DataFrame,
    slot: int,
    model_spec: ModelSpec,
    cloud_host: Optional[str],
    allowed_graph_feats,
    start_row: int = 0,
    max_rows: int = 0,
) -> pd.DataFrame:
    total = len(df)
    start = max(0, min(total, start_row))
    end = total if max_rows <= 0 else min(total, start + max_rows)

    for r in range(start, end):
        message = _strip(df.at[r, 'message'])
        topic = _strip(df.at[r, 'topic'])
        if not message:
            continue

        if _strip(df.at[r, col('LLm', slot)]):
            continue

        signals = extract_graph_signals(df.iloc[r], allowed=allowed_graph_feats) if allowed_graph_feats else None
        prompt = build_prompt(message=message, topic=topic, graph_signals=signals)
        payload, err, elapsed_s = run_inference(model_spec, prompt, cloud_host=cloud_host)

        df.at[r, col('LLm', slot)] = model_spec.name
        if payload is not None:
            df.at[r, col('Экономический эффект', slot)] = payload['economic_effect']
            df.at[r, col('Информационный резонанс', slot)] = payload['information_resonance']
            df.at[r, col('Правильность определения темы', slot)] = payload['topic_agreement']
            df.at[r, col('Экономический нарратив', slot)] = payload['economic_narrative']
            df.at[r, col('Сила нарратива', slot)] = payload['narrative_strength']
            df.at[r, col('Комментарий', slot)] = payload['comment']
        else:
            df.at[r, col('Комментарий', slot)] = err or 'Unknown error'

        tp = f' | {elapsed_s:.2f}s' if elapsed_s is not None else ''
        status = 'OK' if payload is not None else 'ERR'

        err_part = ''
        if payload is None and err:
            err_part = f' | {err[:120]}'

        print(f'Row {r+1}/{total} | slot {slot} | {model_spec.name} | {status}{tp}{err_part}')

        if model_spec.provider == 'cloud_ollama' and CLOUD_THROTTLE_S > 0:
            time.sleep(CLOUD_THROTTLE_S)

    return df


In [4]:
# Быстрая проверка подключения к Ollama (Cloud API и Local)

from ollama import Client, chat

def _check_ollama(host: Optional[str], label: str) -> None:
    try:
        headers = None
        if host and OLLAMA_API_KEY:
            headers = {'Authorization': f'Bearer {OLLAMA_API_KEY}'}

        c = Client(host=host, headers=headers) if host else Client()
        res = c.list()
        models = res.get('models', []) if isinstance(res, dict) else []
        print(f'[{label}] OK | models: {len(models)}')
    except Exception as e:
        print(f'[{label}] ERR | {type(e).__name__}: {e}')

_check_ollama(OLLAMA_CLOUD_HOST, 'cloud')
_check_ollama(None, 'local')


[cloud] OK | models: 0
[local] OK | models: 0


In [12]:
df = pd.read_excel("E:/Нарративы/Сравнительный анализ llm/Датасет для разметки_filled_20260120_214653.xlsx", sheet_name=SHEET)

In [15]:
df = pd.read_excel(input_path, sheet_name=SHEET)
df = ensure_columns(df, n=5)
df.to_excel(out_path, index=False)
print('Loaded:', input_path)
print('Output:', out_path)


Loaded: e:\Нарративы\Сравнительный анализ llm\Датасет для разметки.xlsx
Output: e:\Нарративы\Сравнительный анализ llm\Датасет для разметки_filled_20260121_195139.xlsx


In [31]:
# Slot 1: run via local Ollama using `from ollama import chat` (no cloud API key required)
model_1 = ModelSpec('gpt-oss:20b-cloud', 'cloud_ollama')
df = run_slot(df, slot=1, model_spec=model_1, cloud_host=OLLAMA_CLOUD_HOST, start_row=START_ROW, max_rows=MAX_ROWS)
df.to_excel(out_path, index=False)
print('Saved:', out_path)


Row 1/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 2/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 3/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 4/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 5/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 6/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 7/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 8/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 9/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 10/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 11/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 12/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 13/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 14/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 15/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 16/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 17/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 18/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 19/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 20/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 21/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 22/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 23/500 | slot 1

KeyboardInterrupt: 

In [ ]:
model_2 = ModelSpec('gpt-oss:120b-cloud', 'cloud_ollama')
df = run_slot(df, slot=2, model_spec=model_2, cloud_host=OLLAMA_CLOUD_HOST, start_row=START_ROW, max_rows=MAX_ROWS)
df.to_excel(out_path, index=False)
print('Saved:', out_path)


Row 1/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 2/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 3/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 4/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 5/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 6/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 7/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 8/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 9/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 10/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 11/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 12/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 13/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 14/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 15/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 16/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 17/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 18/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 19/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 20/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 21/500 | slot 2 | gpt-oss:120b-cloud | OK
Row 22/500 | slot 2 | gpt-oss:120b-cloud | 

In [ ]:
model_3 = ModelSpec('qwen3-vl:235b-instruct-cloud', 'cloud_ollama') 
df = run_slot(df, slot=3, model_spec=model_3, cloud_host=OLLAMA_CLOUD_HOST, start_row=START_ROW, max_rows=MAX_ROWS) 
df.to_excel(out_path, index=False) 
print('Saved:', out_path)

In [ ]:
model_4 = ModelSpec('gemma3:27b-cloud', 'cloud_ollama') 
df = run_slot(df, slot=4, model_spec=model_4, cloud_host=OLLAMA_CLOUD_HOST, start_row=START_ROW, max_rows=MAX_ROWS) 
df.to_excel(out_path, index=False) 
print('Saved:', out_path)

Row 1/500 | slot 4 | gemma3:27b-cloud | OK
Row 2/500 | slot 4 | gemma3:27b-cloud | OK
Row 3/500 | slot 4 | gemma3:27b-cloud | OK
Row 4/500 | slot 4 | gemma3:27b-cloud | OK
Row 5/500 | slot 4 | gemma3:27b-cloud | OK
Row 6/500 | slot 4 | gemma3:27b-cloud | OK
Row 7/500 | slot 4 | gemma3:27b-cloud | OK
Row 8/500 | slot 4 | gemma3:27b-cloud | OK
Row 9/500 | slot 4 | gemma3:27b-cloud | OK
Row 10/500 | slot 4 | gemma3:27b-cloud | OK
Row 11/500 | slot 4 | gemma3:27b-cloud | OK
Row 12/500 | slot 4 | gemma3:27b-cloud | OK
Row 13/500 | slot 4 | gemma3:27b-cloud | OK
Row 14/500 | slot 4 | gemma3:27b-cloud | OK
Row 15/500 | slot 4 | gemma3:27b-cloud | OK
Row 16/500 | slot 4 | gemma3:27b-cloud | OK
Row 17/500 | slot 4 | gemma3:27b-cloud | OK
Row 18/500 | slot 4 | gemma3:27b-cloud | OK
Row 19/500 | slot 4 | gemma3:27b-cloud | OK
Row 20/500 | slot 4 | gemma3:27b-cloud | OK
Row 21/500 | slot 4 | gemma3:27b-cloud | OK
Row 22/500 | slot 4 | gemma3:27b-cloud | OK
Row 23/500 | slot 4 | gemma3:27b-cloud | 

: 

# new

In [3]:
import os, json, re, time
from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple, Set
import pandas as pd
out_path = str(DATA_DIR / f'{name}_filled_{stamp}{ext}')
# ---------- helpers ----------
def _strip(x: Any) -> str:
    if x is None:
        return ''
    return str(x).strip()

def _json_from_text(text: str) -> Dict[str, Any]:
    text = _strip(text)
    if not text:
        raise ValueError('Empty text')

    try:
        obj = json.loads(text)
        if not isinstance(obj, dict):
            raise ValueError('JSON root is not an object')
        return obj
    except Exception:
        pass

    m = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if not m:
        raise ValueError('No JSON object found in model output')

    obj = json.loads(m.group(0))
    if not isinstance(obj, dict):
        raise ValueError('JSON root is not an object')
    return obj

def _to_int(v: Any) -> int:
    if isinstance(v, bool):
        raise ValueError('Boolean is not allowed')
    if isinstance(v, (int, float)):
        return int(v)
    s = _strip(v)
    if not s:
        raise ValueError('Empty number')
    return int(float(s))

def _validate_payload(obj: Dict[str, Any]) -> Dict[str, Any]:
    required = {
        'economic_effect',
        'information_resonance',
        'topic_agreement',
        'economic_narrative',
        'narrative_strength',
        'comment',
    }
    missing = sorted(required - set(obj.keys()))
    if missing:
        raise ValueError('Missing keys: ' + ', '.join(missing))

    payload = {
        'economic_effect': _to_int(obj.get('economic_effect')),
        'information_resonance': _to_int(obj.get('information_resonance')),
        'topic_agreement': _to_int(obj.get('topic_agreement')),
        'narrative_strength': _to_int(obj.get('narrative_strength')),
        'economic_narrative': _strip(obj.get('economic_narrative')),
        'comment': _strip(obj.get('comment')),
    }

    if payload['economic_narrative'] not in {'Да', 'Нет'}:
        raise ValueError('economic_narrative must be "Да" or "Нет"')
    if payload['economic_effect'] < -2 or payload['economic_effect'] > 2:
        raise ValueError('economic_effect out of range (-2..2)')
    for k in ['information_resonance', 'topic_agreement', 'narrative_strength']:
        if payload[k] < 1 or payload[k] > 3:
            raise ValueError(f'{k} out of range (1..3)')

    if not payload['comment']:
        payload['comment'] = ''

    return payload

# ---------- graph formatting ----------
GRAPH_COLS = [
    "msg_sim_degree",
    "msg_clustering",
    "ch_pagerank",
    "ch_out_degree",
    "nbr_hist_viral_share",
]

def format_graph_block(signals: Dict[str, Any]) -> str:
    if not signals:
        return ""
    lines = []
    for k, v in signals.items():
        vv = round(v, 6) if isinstance(v, float) else v
        lines.append(f"- {k}: {vv}")
    return "\n\nДоп. контекст (Graph signals):\n" + "\n".join(lines) + "\n"

def extract_graph_signals(row: pd.Series, allowed: Optional[Set[str]] = None) -> Dict[str, Any]:
    signals = {}
    for c in GRAPH_COLS:
        if allowed is not None and c not in allowed:
            continue
        if c in row.index and pd.notna(row[c]):
            v = row[c]
            signals[c] = float(v) if isinstance(v, (int, float)) else v
    return signals

# ---------- prompt (НЕ МЕНЯЮ ТЕКСТ, только добавляю опциональный параметр) ----------
def build_prompt(message: str, topic: str, graph_signals: Optional[Dict[str, Any]] = None) -> str:
    graph_block = format_graph_block(graph_signals or {})
    return (
        'Ты — эксперт по экономическим новостям и общественному восприятию в России. '
        'Твоя задача — определить, является ли короткая новость экономическим нарративом для широкой российской аудитории.\n\n'
        'Критерии нарратива в вашем понимании:\n\n'
        '1. Релевантность для России: Новость должна иметь прямой или косвенный экономический эффект для жителей России. '
        'Мировые новости без последствий для России — не нарратив.\n\n'
        '2. Широкий общественный резонанс: Фокус на обычных гражданах, а не на узких группах. '
        'Нарратив — это яркая новость, которая может вызвать сильный отклик в массах, желание делиться ею '
        'и влиять на импульсивные решения (например, срочные покупки, вывод средств).\n\n'
        '3. Яркость и сила события: В основе нарратива лежит сильное событие (резкий рост цен, важное политическое заявление, '
        'масштабные санкции), а не рутинная информация.\n\n'
        'Проанализируй новость по следующему плану (рассуждения держи в уме, в ответ не выноси):\n\n'
        '[1] Триггер и релевантность:\n'
        'В чём суть новости? Есть ли чёткое триггерное событие (заявление, решение, кризис)?\n'
        'Имеет ли это событие прямые последствия для экономического положения или настроений широких слоёв населения России?\n\n'
        '[2] Эмоциональный заряд и упрощение:\n'
        'Какие эмоции может вызвать текст у обычного человека? (тревога, страх, гнев, оптимизм).\n'
        'Сводится ли основная мысль к простым, обобщающим формулировкам? (Например: «Цены на всё вырастут», «Рубль обвалится», «Наступит дефицит»).\n\n'
        '[3] Логика воздействия и аудитория:\n'
        'Пытается ли новость объяснить, как именно это событие повлияет на жизнь людей, а не просто констатирует факт?\n'
        'Направлена ли новость на массовую аудиторию, а не на профессионалов?\n\n'
        '[4] Источники и резонансный потенциал:\n'
        'Кто является источником или героем новости? (Правительство, ЦБ, известный политик, эксперты в СМИ). '
        'Усиливает ли это авторитетность и потенциальное распространение?\n'
        'Может ли эта новость стать «вирусной» историей для обсуждения в соцсетях и бытовых разговорах?\n\n'
        'Пояснения к полям:\n\n'
        'economic_narrative (Да/Нет): Итоговое решение. Да — если есть триггерное событие, релевантное для широкой российской аудитории, '
        'и новость обладает потенциалом вызвать эмоциональный отклик и массовое обсуждение.\n\n'
        'narrative_strength (1-3): Сила нарративных свойств. Оценивай, насколько текст эмоционален, упрощён и побуждает к действию. '
        '1 — констатация факта; 3 — прямое предупреждение о катастрофических последствиях для всех. '
        'Не нарративные новости в большинстве своем должны получать оценку 1, при этом нарративы тоже могут изредка получать такую оценку.\n\n'
        'economic_effect (-2..2): Влияние на экономическое положение обычного человека в России. Примеры: крах банков: -2, рост в отдельном регионе: -1, отмена санкций: 2.\n\n'
        'topic_agreement (1-3): Твоя оценка правильности выбранной темы (topic). Считай, что тема выбрана из фиксированного списка.\n\n'
        'information_resonance (1-3): Потенциал широкого и эмоционального восприятия в обществе. Высокий резонанс — темы, затрагивающие каждого '
        '(цены на еду, бензин, рубль, важные политические решения для всего населения).\n\n'
        'ВАЖНО: Ответь строго одним JSON-объектом БЕЗ пояснений, БЕЗ markdown и БЕЗ текста вокруг.\n'
        'Схема JSON (ключи строго такие):\n'
        '{\n'
        '  "economic_effect": -2,\n'
        '  "information_resonance": 1,\n'
        '  "topic_agreement": 1,\n'
        '  "economic_narrative": "Да",\n'
        '  "narrative_strength": 1,\n'
        '  "comment": "..."\n'
        '}\n\n'
        f'Тема (topic): {topic}\n'
        f'Новость (message): {message}\n'
        f'{graph_block}'
        )
        

# ---------- model infra ----------
@dataclass(frozen=True)
class ModelSpec:
    name: str
    provider: str

CLOUD_TIMEOUT_S = 60
NUM_PREDICT = 1024
RETRY_ATTEMPTS = 3
RETRY_BACKOFF_S = 1.0
CLOUD_THROTTLE_S = 0.25

def _normalize_ollama_model_name(model: str, host: Optional[str]) -> str:
    if host and model.endswith('-cloud'):
        return model[:-6]
    return model

def _ollama_client(host: str, api_key: str):
    from ollama import Client
    headers = {'Authorization': f'Bearer {api_key}'}
    return Client(host=host, headers=headers, timeout=CLOUD_TIMEOUT_S)

def _ollama_message_content(resp: Any) -> str:
    if resp is None:
        return ''
    if isinstance(resp, dict):
        msg = resp.get('message') or {}
        if isinstance(msg, dict):
            return _strip(msg.get('content'))
    if hasattr(resp, 'message') and hasattr(resp.message, 'content'):
        return _strip(resp.message.content)
    try:
        msg = resp['message']
        return _strip(msg['content'])
    except Exception:
        return _strip(str(resp))

def _should_retry_exc(e: Exception) -> bool:
    status = getattr(e, 'status_code', None)
    if isinstance(status, int) and status in {408, 409, 425, 429, 500, 502, 503, 504}:
        return True
    msg = str(e).lower()
    return any(s in msg for s in ['empty model response', 'timed out', 'timeout', 'connection', 'network'])

def infer_ollama(model: str, prompt: str, host: str, api_key: str) -> str:
    last_err: Optional[Exception] = None
    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            client = _ollama_client(host, api_key)
            resp = client.chat(
                model=_normalize_ollama_model_name(model, host),
                messages=[{'role': 'user', 'content': prompt}],
                options={'temperature': 0.2, 'num_predict': NUM_PREDICT},
            )
            text = _ollama_message_content(resp)
            if not text:
                raise ValueError('Empty model response')
            return text
        except Exception as e:
            last_err = e
            if attempt < RETRY_ATTEMPTS and _should_retry_exc(e):
                time.sleep(RETRY_BACKOFF_S * attempt)
                continue
            break
    raise last_err or ValueError('Empty model response')

def run_inference(model_spec: ModelSpec, prompt: str, cloud_host: str, api_key: str) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
    try:
        text = infer_ollama(model_spec.name, prompt, host=cloud_host, api_key=api_key)
        payload = _validate_payload(_json_from_text(text))
        return payload, None
    except Exception as e:
        return None, f'{type(e).__name__}: {e}'

def col(name: str, idx: int) -> str:
    return f'{name} ({idx})'

def ensure_columns(df: pd.DataFrame, n: int) -> pd.DataFrame:
    for c in ['message', 'topic']:
        if c not in df.columns:
            raise ValueError(f'Missing required column: {c}')
    metric_cols = [
        'LLm', 'Экономический эффект', 'Информационный резонанс',
        'Правильность определения темы', 'Экономический нарратив', 'Сила нарратива', 'Комментарий',
    ]
    for i in range(1, n + 1):
        for m in metric_cols:
            cn = col(m, i)
            if cn not in df.columns:
                df[cn] = ''
            df[cn] = df[cn].astype('object').where(~pd.isna(df[cn]), '')
    return df

def run_slot(
    df: pd.DataFrame,
    slot: int,
    model_spec: ModelSpec,
    cloud_host: str,
    api_key = os.getenv('OLLAMA_API_KEY'),
    allowed_graph_feats: Optional[Set[str]] = None,
    start_row: int = 0,
    max_rows: int = 0,
) -> pd.DataFrame:
    total = len(df)
    start = max(0, min(total, start_row))
    end = total if max_rows <= 0 else min(total, start + max_rows)

    for r in range(start, end):
        message = _strip(df.at[r, 'message'])
        topic = _strip(df.at[r, 'topic'])
        if not message:
            continue
        if _strip(df.at[r, col('LLm', slot)]):
            continue

        signals = extract_graph_signals(df.iloc[r], allowed=allowed_graph_feats) if allowed_graph_feats else None
        prompt = build_prompt(message=message, topic=topic, graph_signals=signals)

        payload, err = run_inference(model_spec, prompt, cloud_host=cloud_host, api_key=api_key)

        df.at[r, col('LLm', slot)] = model_spec.name
        if payload is not None:
            df.at[r, col('Экономический эффект', slot)] = payload['economic_effect']
            df.at[r, col('Информационный резонанс', slot)] = payload['information_resonance']
            df.at[r, col('Правильность определения темы', slot)] = payload['topic_agreement']
            df.at[r, col('Экономический нарратив', slot)] = payload['economic_narrative']
            df.at[r, col('Сила нарратива', slot)] = payload['narrative_strength']
            df.at[r, col('Комментарий', slot)] = payload['comment']
            status = "OK"
        else:
            df.at[r, col('Комментарий', slot)] = err or 'Unknown error'
            status = "ERR"

        print(f'Row {r+1}/{total} | slot {slot} | {model_spec.name} | {status}')
        time.sleep(CLOUD_THROTTLE_S)

    return df


In [4]:
# =========================
# === NEW: paths for graph + messages ===
# =========================
from pathlib import Path

# DATA_DIR / ART_DIR already set in bootstrap cell
MESSAGES_CSV = str(DATA_DIR / 'messages_with_reactions.csv')
GRAPH_FEATS_CSV = str(ART_DIR / "graph_features.csv")

# === NEW: switch ===
USE_GRAPH_SIGNALS = True

# (рекомендую) ключ только из env
OLLAMA_API_KEY = os.getenv('OLLAMA_API_KEY')


In [5]:
# =========================
# === NEW: graph signals helpers ===
# =========================
GRAPH_COLS = [
    "msg_sim_degree",
    "msg_clustering",
    "ch_pagerank",
    "ch_out_degree",
    "nbr_hist_viral_share",
]


def format_graph_block(signals: Dict[str, Any]) -> str:
    if not signals:
        return ""
    lines = []
    for k, v in signals.items():
        vv = round(v, 6) if isinstance(v, float) else v
        lines.append(f"- {k}: {vv}")
    return "\n\nДоп. контекст (Graph signals):\n" + "\n".join(lines) + "\n"


In [6]:
# =========================
# === NEW: attach id + graph_features to df ===
# =========================
def attach_ids_and_graph_features(df: pd.DataFrame) -> pd.DataFrame:
    # load sources
    msgs = pd.read_csv(MESSAGES_CSV, usecols=["id", "message", "id_channel", "date"], low_memory=False)
    msgs = msgs.dropna(subset=["id", "message"]).copy()
    msgs["id"] = msgs["id"].astype(str)
    msgs["message_norm"] = msgs["message"].astype(str).str.strip()

    graph_features = pd.read_csv(GRAPH_FEATS_CSV)
    graph_features["id"] = graph_features["id"].astype(str)

    # normalize excel
    df = df.copy()
    df["message_norm"] = df["message"].astype(str).str.strip()

    # 1) exact match only for unique texts
    amb = msgs.groupby("message_norm")["id"].nunique().reset_index(name="n_ids")
    unique_texts = amb[amb["n_ids"] == 1][["message_norm"]]
    map_unique = (
        msgs.merge(unique_texts, on="message_norm", how="inner")
            .drop_duplicates(subset=["message_norm"])[["message_norm", "id"]]
    )
    df = df.merge(map_unique, on="message_norm", how="left")

    # 2) prefix fallback for remaining NaNs (safe, best-effort)
    tmp = msgs[["id", "message"]].copy()

    def _fill_id_by_prefix(row_idx: int, n: int = 80):
        if pd.notna(df.at[row_idx, "id"]):
            return
        prefix = str(df.at[row_idx, "message"])[:n]
        m = tmp[tmp["message"].astype(str).str.contains(prefix, regex=False, na=False)]
        if len(m) > 0:
            df.at[row_idx, "id"] = str(m.iloc[0]["id"])

    for i in df.index[df["id"].isna()].tolist():
        _fill_id_by_prefix(i, n=80)

    # 3) merge graph features
    df = df.merge(graph_features, on="id", how="left")

    print("matched ids:", df["id"].notna().mean(), "| unmatched:", int(df["id"].isna().sum()))
    if "msg_sim_degree" in df.columns:
        print("share with graph features:", df["msg_sim_degree"].notna().mean())

    return df


In [7]:
df = pd.read_excel(input_path, sheet_name=SHEET)
df = ensure_columns(df, n=5)


In [8]:
# === NEW ===
df = attach_ids_and_graph_features(df)


matched ids: 1.0 | unmatched: 0
share with graph features: 1.0


In [9]:
df.head()

,message,topic,LLm (1),Экономический эффект (1),Информационный резонанс (1),Правильность определения темы (1),Экономический нарратив (1),Сила нарратива (1),Комментарий (1),LLm (2),...,Сила нарратива (5),Комментарий (5),message_norm,id,id_channel,msg_sim_degree,ch_out_degree,ch_pagerank,msg_clustering,nbr_hist_viral_share
0,#POSI \nГК Позитив — лидер в рейтинге покупок ...,Рынки капитала,,,,,,,,,...,,,#POSI \nГК Позитив — лидер в рейтинге покупок ...,8c497c96-be46-4bb4-888f-25e40d20debb,2,0.0,11724,0.043697,0.0,0.0
1,⚡️Россельхознадзор опроверг запрет на ввоз ман...,Международная торговля,,,,,,,,,...,,,⚡️Россельхознадзор опроверг запрет на ввоз ман...,63c159f3-4205-4477-bb06-e00c0bde9187,1,0.0,23522,0.087665,0.0,0.0
2,Корпоративные бонды с доходностью 20-21%+ — се...,Рынки капитала,,,,,,,,,...,,,Корпоративные бонды с доходностью 20-21%+ — се...,9d73c104-aa5e-478d-bc27-d57f15c0c646,2,0.0,11724,0.043697,0.0,0.0
3,#MTLR\n🪨 Мечел в 2025г может сократить отгрузк...,Корпоративные финансы,,,,,,,,,...,,,#MTLR\n🪨 Мечел в 2025г может сократить отгрузк...,db2e74c1-fefa-4333-823c-6e0a527e87a0,2,1.0,11724,0.043697,0.0,0.0
4,Россия вошла в топ-5 стран по оттоку миллионер...,Макроэкономика,,,,,,,,,...,,,Россия вошла в топ-5 стран по оттоку миллионер...,031d93c6-89f0-426a-ada0-9f6f4174f4b7,18,0.0,12549,0.046771,0.0,0.0


In [10]:
df.to_excel(str(DATA_DIR / 'markup_dataset_with_ids.xlsx'))


In [11]:
# какие фичи разрешены вообще (колонки в df после merge)
GRAPH_COLS = [
    "msg_sim_degree",
    "msg_clustering",
    "ch_pagerank",
    "ch_out_degree",
    "nbr_hist_viral_share",
]

ABLATION = {
    1: set(),  # без графа
    2: {"msg_sim_degree"},
    3: {"msg_sim_degree", "msg_clustering"},
    4: {"msg_sim_degree", "msg_clustering", "ch_pagerank"},
    5: {"msg_sim_degree", "msg_clustering", "ch_pagerank", "nbr_hist_viral_share"},
}


In [12]:
def extract_graph_signals(row: pd.Series, allowed: Optional[set] = None) -> Dict[str, Any]:
    signals = {}
    for c in GRAPH_COLS:
        if allowed is not None and c not in allowed:
            continue
        if c in row.index and pd.notna(row[c]):
            v = row[c]
            signals[c] = float(v) if isinstance(v, (int, float)) else v
    return signals


In [13]:
df = ensure_columns(df, n=5)


In [20]:
MODEL = ModelSpec('gpt-oss:20b-cloud', 'cloud_ollama')

for slot, feats in ABLATION.items():
    df = run_slot(
        df,
        slot=slot,
        model_spec=MODEL,
        cloud_host=OLLAMA_CLOUD_HOST,
        allowed_graph_feats=feats,
        start_row=START_ROW,
        max_rows=MAX_ROWS,
    )
    df.to_excel(out_path, index=False)
    print("Saved:", out_path)


Row 3/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 4/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 5/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 6/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 7/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 8/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 9/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 10/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 11/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 12/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 13/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 14/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 15/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 16/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 17/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 18/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 19/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 20/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 21/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 22/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 23/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 24/500 | slot 1 | gpt-oss:20b-cloud | OK
Row 25/500 | slot

In [ ]:
# 37*250/60/60

2.569444444444444

In [ ]:
# ABLATION = {
#   1: set(),  # без графа
#   2: {"msg_sim_degree"},
#   3: {"msg_sim_degree", "msg_clustering"},
#   4: {"msg_sim_degree", "msg_clustering", "ch_pagerank"},
#   5: {"msg_sim_degree", "msg_clustering", "ch_pagerank", "nbr_hist_viral_share"},
# }


# metrics

In [23]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
)

# ======================
# CONFIG
# ======================
PRED_XLSX = str(DATA_DIR / "markup_dataset_filled_20260220.xlsx")
GOLD_CSV = str(DATA_DIR / "golden_set_v1.csv")  # экспорт из hf_datasets/economic-narratives-golden-set

SLOTS = [1, 2, 3, 4, 5]

# если хочешь красивый label (не обязательно)
ABLATION = {
    1: "no_graph",
    2: "msg_sim_degree",
    3: "msg_sim_degree+msg_clustering",
    4: "msg_sim_degree+msg_clustering+ch_pagerank",
    5: "msg_sim_degree+msg_clustering+ch_pagerank+nbr_hist_viral_share",
}

# ======================
# HELPERS: parsing + mapping
# ======================
def _strip(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return str(x).strip()

def _to_num(x):
    """robust numeric parse -> float or NaN"""
    s = _strip(x)
    if s == "":
        return np.nan
    try:
        return float(s.replace(",", "."))
    except Exception:
        return np.nan

def map_yesno_to_01(x):
    """
    Pred: 'Да'/'Нет' or 1/0
    Gold: 0/1 (может быть float)
    """
    s = _strip(x).lower()
    if s in {"да", "1", "true", "yes"}:
        return 1
    if s in {"нет", "0", "false", "no"}:
        return 0
    # если вдруг число
    v = _to_num(x)
    if np.isnan(v):
        return np.nan
    return int(round(v))

def map_1_3_to_0_05_1(x):
    """LLM: 1..3 -> {0, 0.5, 1}"""
    v = _to_num(x)
    if np.isnan(v):
        return np.nan
    v = int(round(v))
    if v == 1:
        return 0.0
    if v == 2:
        return 0.5
    if v == 3:
        return 1.0
    return np.nan

def map_effect_minus2_2_to_minus1_1(x):
    """LLM: -2..2 -> {-1,-0.5,0,0.5,1}"""
    v = _to_num(x)
    if np.isnan(v):
        return np.nan
    # ожидаем int, но на всякий
    v = int(round(v))
    if v < -2 or v > 2:
        return np.nan
    return v / 2.0

def safe_reg_metrics(y_true, y_pred):
    """MAE/RMSE + Spearman for numeric targets"""
    mask = (~pd.isna(y_true)) & (~pd.isna(y_pred))
    y_t = np.array(y_true[mask], dtype=float)
    y_p = np.array(y_pred[mask], dtype=float)
    if len(y_t) == 0:
        return {"n": 0, "mae": np.nan, "rmse": np.nan, "spearman": np.nan}
    mae = mean_absolute_error(y_t, y_p)
    rmse = mean_squared_error(y_t, y_p) ** 0.5
    # spearman через pandas (без scipy)
    spearman = pd.Series(y_t).corr(pd.Series(y_p), method="spearman")
    return {"n": int(len(y_t)), "mae": mae, "rmse": rmse, "spearman": spearman}

# ======================
# LOAD GOLD
# ======================
gold = pd.read_csv(GOLD_CSV, low_memory=False)

# стандартизация колонок голда
# ожидаем: message_id, economic_narrative, narrative_strength, economic_effect, information_resonance
gold = gold.copy()
gold["message_id"] = gold["message_id"].astype(str)

gold["gold_economic_narrative"] = gold["economic_narrative"].apply(map_yesno_to_01).astype("Int64")
gold["gold_strength"] = pd.to_numeric(gold["narrative_strength"], errors="coerce")
gold["gold_effect"] = pd.to_numeric(gold["economic_effect"], errors="coerce")
gold["gold_resonance"] = pd.to_numeric(gold["information_resonance"], errors="coerce")

# ======================
# LOAD PRED FILE (EXCEL)
# ======================
pred = pd.read_excel(PRED_XLSX)
pred = pred.copy()

# ключ в твоём excel после attach_ids_and_graph_features
# (если вдруг у тебя колонка называется message_id — просто поменяй тут)
if "id" not in pred.columns:
    raise ValueError('Expected column "id" in prediction Excel (after attach_ids_and_graph_features).')
pred["id"] = pred["id"].astype(str)

def col(name, slot):
    return f"{name} ({slot})"

# ======================
# BUILD LONG TABLE: one row = one (message_id, config/slot)
# ======================
rows = []
for slot in SLOTS:
    r = pd.DataFrame({
        "message_id": pred["id"],
        "slot": slot,
        "ablation": ABLATION.get(slot, f"slot_{slot}"),
        "model": pred.get(col("LLm", slot), ""),
        "pred_narrative_raw": pred.get(col("Экономический нарратив", slot), ""),
        "pred_strength_raw": pred.get(col("Сила нарратива", slot), np.nan),
        "pred_effect_raw": pred.get(col("Экономический эффект", slot), np.nan),
        "pred_resonance_raw": pred.get(col("Информационный резонанс", slot), np.nan),
        "pred_comment": pred.get(col("Комментарий", slot), ""),
    })

    # normalize scales to match GOLD
    r["pred_economic_narrative"] = r["pred_narrative_raw"].apply(map_yesno_to_01).astype("Int64")
    r["pred_strength"] = r["pred_strength_raw"].apply(map_1_3_to_0_05_1)
    r["pred_effect"] = r["pred_effect_raw"].apply(map_effect_minus2_2_to_minus1_1)
    r["pred_resonance"] = r["pred_resonance_raw"].apply(map_1_3_to_0_05_1)

    # конфиг-лейбл: модель + абляция
    r["model"] = r["model"].astype(str).replace({"nan": ""}).fillna("")
    r["config"] = np.where(
        r["model"].str.strip() != "",
        r["model"].str.strip() + " | " + r["ablation"],
        "UNKNOWN_MODEL | " + r["ablation"]
    )

    rows.append(r)

pred_long = pd.concat(rows, ignore_index=True)

# ======================
# MERGE WITH GOLD
# ======================
m = pred_long.merge(gold, on="message_id", how="inner")

# если хочешь видеть coverage:
coverage = pred_long.merge(gold[["message_id"]], on="message_id", how="left", indicator=True)["_merge"].value_counts()
print("Merge coverage (pred_long vs gold):")
print(coverage)

# ======================
# METRICS PER CONFIG
# ======================
def eval_config(df_cfg: pd.DataFrame) -> dict:
    out = {
        "config": df_cfg["config"].iloc[0],
        "slot": int(df_cfg["slot"].iloc[0]),
        "ablation": df_cfg["ablation"].iloc[0],
        "model": df_cfg["model"].iloc[0],
        "n_gold_matched": int(len(df_cfg)),
    }

    # --- narrative (binary) ---
    y_true = df_cfg["gold_economic_narrative"].astype("float")
    y_pred = df_cfg["pred_economic_narrative"].astype("float")
    mask = (~pd.isna(y_true)) & (~pd.isna(y_pred))
    y_t = y_true[mask].astype(int)
    y_p = y_pred[mask].astype(int)
    out["n_narr_eval"] = int(mask.sum())

    if out["n_narr_eval"] > 0:
        acc = accuracy_score(y_t, y_p)
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_t, y_p, average="binary", pos_label=1, zero_division=0
        )
        cm = confusion_matrix(y_t, y_p, labels=[0, 1])
        out.update({
            "narr_acc": acc,
            "narr_prec": prec,
            "narr_rec": rec,
            "narr_f1": f1,
            "cm_tn": int(cm[0, 0]),
            "cm_fp": int(cm[0, 1]),
            "cm_fn": int(cm[1, 0]),
            "cm_tp": int(cm[1, 1]),
        })
    else:
        out.update({"narr_acc": np.nan, "narr_prec": np.nan, "narr_rec": np.nan, "narr_f1": np.nan})

    # --- numeric targets (all rows) ---
    s = safe_reg_metrics(df_cfg["gold_strength"], df_cfg["pred_strength"])
    e = safe_reg_metrics(df_cfg["gold_effect"], df_cfg["pred_effect"])
    r = safe_reg_metrics(df_cfg["gold_resonance"], df_cfg["pred_resonance"])

    out.update({
        "strength_n": s["n"], "strength_mae": s["mae"], "strength_rmse": s["rmse"], "strength_spearman": s["spearman"],
        "effect_n": e["n"],     "effect_mae": e["mae"],     "effect_rmse": e["rmse"],     "effect_spearman": e["spearman"],
        "reson_n": r["n"],      "reson_mae": r["mae"],      "reson_rmse": r["rmse"],      "reson_spearman": r["spearman"],
    })

    # --- optional: metrics only on gold narratives == 1 (часто полезнее) ---
    df_pos = df_cfg[df_cfg["gold_economic_narrative"] == 1]
    sp = safe_reg_metrics(df_pos["gold_strength"], df_pos["pred_strength"])
    ep = safe_reg_metrics(df_pos["gold_effect"], df_pos["pred_effect"])
    rp = safe_reg_metrics(df_pos["gold_resonance"], df_pos["pred_resonance"])
    out.update({
        "POS_strength_n": sp["n"], "POS_strength_mae": sp["mae"],
        "POS_effect_n": ep["n"],   "POS_effect_mae": ep["mae"],
        "POS_reson_n": rp["n"],    "POS_reson_mae": rp["mae"],
    })

    return out

metrics = []
for cfg, g in m.groupby("config", sort=False):
    metrics.append(eval_config(g))

metrics_df = pd.DataFrame(metrics)

# ранжирование: основной таргет — F1 по narrative, тай-брейк — recall, потом MAE силы нарратива
metrics_df = metrics_df.sort_values(
    by=["narr_f1", "narr_rec", "strength_mae"],
    ascending=[False, False, True],
).reset_index(drop=True)

print("\n=== LEADERBOARD ===")
print(metrics_df[[
    "config", "n_gold_matched",
    "narr_f1", "narr_prec", "narr_rec", "narr_acc",
    "strength_mae", "effect_mae", "reson_mae",
    "POS_strength_mae", "POS_effect_mae", "POS_reson_mae",
]])

# сохранить
metrics_df.to_csv("llm_vs_golden_metrics.csv", index=False)
m.to_csv("llm_vs_golden_joined_rows.csv", index=False)

print("\nSaved: llm_vs_golden_metrics.csv")
print("Saved: llm_vs_golden_joined_rows.csv")


Merge coverage (pred_long vs gold):
_merge
both          2495
left_only        5
right_only       0
Name: count, dtype: int64

=== LEADERBOARD ===
                                              config  n_gold_matched  \
0                       gpt-oss:20b-cloud | no_graph             499   
1  gpt-oss:20b-cloud | msg_sim_degree+msg_cluster...             499   
2  gpt-oss:20b-cloud | msg_sim_degree+msg_clustering             499   
3                 gpt-oss:20b-cloud | msg_sim_degree             499   
4  gpt-oss:20b-cloud | msg_sim_degree+msg_cluster...             499   

    narr_f1  narr_prec  narr_rec  narr_acc  strength_mae  effect_mae  \
0  0.530769   0.704082  0.425926  0.748454      0.170515    0.228557   
1  0.524345   0.729167  0.409357  0.744980      0.168173    0.241466   
2  0.486891   0.677083  0.380117  0.724900      0.171787    0.237550   
3  0.471910   0.656250  0.368421  0.717435      0.179259    0.234469   
4  0.458498   0.707317  0.339181  0.725451      0.173046    

In [25]:
metrics_df.head(1000)

,config,slot,ablation,model,n_gold_matched,n_narr_eval,narr_acc,narr_prec,narr_rec,narr_f1,...,reson_n,reson_mae,reson_rmse,reson_spearman,POS_strength_n,POS_strength_mae,POS_effect_n,POS_effect_mae,POS_reson_n,POS_reson_mae
0,gpt-oss:20b-cloud | no_graph,1,no_graph,gpt-oss:20b-cloud,499,485,0.748454,0.704082,0.425926,0.530769,...,485,0.235567,0.332787,0.442393,162,0.389198,162,0.274691,162,0.371296
1,gpt-oss:20b-cloud | msg_sim_degree+msg_cluster...,4,msg_sim_degree+msg_clustering+ch_pagerank,gpt-oss:20b-cloud,499,498,0.744980,0.729167,0.409357,0.524345,...,498,0.245181,0.338791,0.400612,171,0.376608,171,0.280409,171,0.400000
2,gpt-oss:20b-cloud | msg_sim_degree+msg_clustering,3,msg_sim_degree+msg_clustering,gpt-oss:20b-cloud,499,498,0.724900,0.677083,0.380117,0.486891,...,498,0.247992,0.345249,0.412842,171,0.374854,171,0.273392,171,0.402924
3,gpt-oss:20b-cloud | msg_sim_degree,2,msg_sim_degree,gpt-oss:20b-cloud,499,499,0.717435,0.656250,0.368421,0.471910,...,499,0.243888,0.346786,0.419749,171,0.395906,171,0.285673,171,0.383041
4,gpt-oss:20b-cloud | msg_sim_degree+msg_cluster...,5,msg_sim_degree+msg_clustering+ch_pagerank+nbr_...,gpt-oss:20b-cloud,499,499,0.725451,0.707317,0.339181,0.458498,...,499,0.250701,0.351236,0.390783,171,0.396491,171,0.272222,171,0.409357


In [26]:
import pandas as pd

pred = pd.read_excel(PRED_XLSX)  # тот же файл с предиктами, где есть graph колонки
gold = pd.read_csv(GOLD_CSV)
gold_ids = set(gold["message_id"].astype(str))

eval_df = pred[pred["id"].astype(str).isin(gold_ids)].copy()

GRAPH_COLS = ["msg_sim_degree","msg_clustering","ch_pagerank","ch_out_degree","nbr_hist_viral_share"]
print("Rows in eval:", len(eval_df))
print("Non-null share per graph col:")
print((eval_df[GRAPH_COLS].notna().mean()).sort_values(ascending=False))

print("\nShare of rows with ANY graph signal:")
print(eval_df[GRAPH_COLS].notna().any(axis=1).mean())


Rows in eval: 499
Non-null share per graph col:
msg_sim_degree          1.0
msg_clustering          1.0
ch_pagerank             1.0
ch_out_degree           1.0
nbr_hist_viral_share    1.0
dtype: float64

Share of rows with ANY graph signal:
1.0


In [27]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# возьми eval_df из твоей проверки (499 строк с граф-колонками) и gold
pred = pd.read_excel(PRED_XLSX)
gold = pd.read_csv(GOLD_CSV)
gold_ids = set(gold["message_id"].astype(str))

eval_df = pred[pred["id"].astype(str).isin(gold_ids)].copy()
g = gold.copy()
g["message_id"] = g["message_id"].astype(str)

m = eval_df.merge(g, left_on="id", right_on="message_id", how="inner")

GRAPH_COLS = ["msg_sim_degree","msg_clustering","ch_pagerank","ch_out_degree","nbr_hist_viral_share"]

y = m["economic_narrative"].astype(float).astype(int).values  # 0/1
print("Class balance (gold narrative=1):", y.mean())

# AUC по каждой фиче отдельно
for c in GRAPH_COLS:
    x = m[c].astype(float).values
    auc = roc_auc_score(y, x)
    print(c, "AUC:", round(auc, 4))

# AUC простой логрег по всем фичам
X = m[GRAPH_COLS].astype(float).values
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500))
clf.fit(X, y)
p = clf.predict_proba(X)[:, 1]
print("LogReg(all graph feats) AUC:", round(roc_auc_score(y, p), 4))

# Spearman с резонансом (если он в голде есть)
if "information_resonance" in m.columns:
    for c in GRAPH_COLS:
        sp = m[c].corr(m["information_resonance"], method="spearman")
        print("Spearman", c, "vs gold_resonance:", round(sp, 4))


Class balance (gold narrative=1): 0.342685370741483
msg_sim_degree AUC: 0.4977
msg_clustering AUC: 0.4994
ch_pagerank AUC: 0.4775
ch_out_degree AUC: 0.4775
nbr_hist_viral_share AUC: 0.5113
LogReg(all graph feats) AUC: 0.5569
Spearman msg_sim_degree vs gold_resonance: 0.0124
Spearman msg_clustering vs gold_resonance: 0.0233
Spearman ch_pagerank vs gold_resonance: -0.0124
Spearman ch_out_degree vs gold_resonance: -0.0124
Spearman nbr_hist_viral_share vs gold_resonance: 0.0705
